In [64]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [65]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd()
data_dir = base_path.parent / "data" / "raw"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

THE PATH

In [66]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

SPLITTING THE DATA INTO X AND 

In [67]:
x = train.drop("SalePrice",axis = 1)
y = train["SalePrice"]

FEATURE ENGINEERING

In [68]:
def add_features(df):
    df = df.copy()
    df["TotalSF"]= df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"]=(df["FullBath"] + 0.5 * df["HalfBath"] +df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
    df["TotalPorch"]=(df["OpenPorchSF"] + df["EnclosedPorch"] +df["3SsnPorch"]+ df["ScreenPorch"])
    df["HouseAge"]=df["YrSold"] - df["YearBuilt"]
    df["RemodAge"]=df["YrSold"] - df["YearRemodAdd"]
    df["GarageAge"]=df["YrSold"] - df["GarageYrBlt"].fillna(df["YearBuilt"])
    df["IsRemodeled"]=(df["YearBuilt"] != df["YearRemodAdd"]).astype(int)
    df["QualxSF"]=df["OverallQual"] * df["TotalSF"]
    df["OverallScore"]=df["OverallQual"] * df["OverallCond"]
    df["AreaQuality"]=df["GrLivArea"] * df["OverallQual"]
    df["AgeQuality"]=df["HouseAge"] * df["OverallQual"]
    df["TotalQuality"]=df["TotalSF"] * df["OverallQual"]
    return df

x    = add_features(x)
test = add_features(test) 

SPLITTING THE DATA INTO NUMRICAL AND CATEGORICAL COLUMNS


In [69]:
num_cols = x.select_dtypes(include=[np.number]).columns.drop(['Id', 'SalePrice'], errors='ignore')
cat_col = x.select_dtypes(include=[np.object_]).columns
num_cols , cat_col

(Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
        'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
        'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
        'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
        'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
        'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
        'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal',
        'MoSold', 'YrSold', 'TotalSF', 'TotalBath', 'TotalPorch', 'HouseAge',
        'RemodAge', 'GarageAge', 'IsRemodeled', 'QualxSF', 'OverallScore',
        'AreaQuality', 'AgeQuality', 'TotalQuality'],
       dtype='object'),
 Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
        'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
        'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
  

DEALING WITH DATA SKEWNESS

SPLITTING THE DATA INTO TRAIN AND TEST


In [70]:
from sklearn.model_selection import train_test_split
x_train , x_val , y_train , y_val = train_test_split(x,y,test_size = 0.2 , random_state= 42)

DEALING WITH OUTLIERS

In [71]:
for i in num_cols:
    inliers = x_train[i].quantile(0.99)
    x_train[i] = x_train[i].clip(upper = inliers)
    test[i] = test[i].clip(upper = inliers)
    x_val[i] = x_val[i].clip(upper = inliers)

CREATING A PIPLINE TO DEAL WITH NUMRICAL DATA

In [72]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([("imputer" , SimpleImputer(strategy='constant' , fill_value = 0)),
                        ('scaler' , StandardScaler())
                        ])

In [73]:
ordinal_col = ['ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure','HeatingQC','FireplaceQu','GarageQual','GarageCond','KitchenQual','PoolQC']

In [74]:
one_hot_col = ['MSZoning','Street','Alley','LotShape','LandContour','Utilities','LotConfig','LandSlope','Neighborhood','Condition2','Condition1','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType','Foundation','BsmtFinType1','BsmtFinType2','Heating','CentralAir','Electrical','Functional','GarageType','GarageFinish','PavedDrive','Fence','MiscFeature','SaleType','SaleCondition']

CREATING A PIPLINE TO DEAL WITH CATEGORICAL DATA USING ONE HOT


In [75]:
from sklearn.preprocessing import OneHotEncoder
one_pipeline = Pipeline([('imputer' , SimpleImputer(strategy="constant",fill_value="None")),
                         ('one_hot',OneHotEncoder(handle_unknown='ignore'))])

CREATING A PIPLINE TO DEAL WITH CATEGORICAL DATA USING ORDINAL ENCODING

In [76]:
from sklearn.preprocessing import OrdinalEncoder
quality_order = ['None','Po', 'Fa', 'TA', 'Gd', 'Ex',]
categories = [quality_order for _ in range(len(ordinal_col))]
ord_pipeline = Pipeline([('imputer' , SimpleImputer(strategy="constant",fill_value="None")),
                         ('ord',OrdinalEncoder(categories =categories ,handle_unknown= "use_encoded_value" , unknown_value=-1))])

COLUMN TRANSFORMER TO APPLY THE PIPLINES ALL TOGETHER THEN TRANSFORM IT TO THE TRAIN AND TEST DATA SETS


In [77]:
from sklearn.compose import ColumnTransformer
full_pipe = ColumnTransformer([('num' , num_pipeline , num_cols ),
              ('cat' , one_pipeline , one_hot_col),
              ('ord' , ord_pipeline , ordinal_col)
              ])
x_train_transformed = full_pipe.fit_transform(x_train)
x_val_transformed = full_pipe.transform(x_val)
test_transformed = full_pipe.transform(test)

LINEAR MODEL

In [78]:
from sklearn.linear_model import LinearRegression
lin_model = LinearRegression()

lin_model.fit(x_train_transformed , y_train)

prediction = lin_model.predict(x_val_transformed)

from sklearn.metrics import mean_squared_error,r2_score

mse = mean_squared_error(y_val, prediction)
r2 = r2_score(y_val, prediction)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error: 4184621268686995413788524544.00
R2 Score: -545559752695522752.0000


RANDOM FOREST MODEL

In [79]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100,max_depth=10,random_state=42)

rf_model.fit(x_train_transformed , y_train)
rf_prediction = rf_model.predict(x_val_transformed)

from sklearn.metrics import mean_squared_error,r2_score

mse = mean_squared_error(y_val, rf_prediction)
r2 = r2_score(y_val, rf_prediction)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error: 899164062.69
R2 Score: 0.8828


XGB MODEL

In [80]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xg_model = XGBRegressor(n_estimators = 1000,
learning_rate = 0.05, 
max_depth=4,           
subsample=0.8,         
colsample_bytree=0.8,  
random_state=42,
n_jobs=-1)

y_train_log = np.log1p(y_train)
xg_model.fit(x_train_transformed, y_train_log)

xg_prediction_log = xg_model.predict(x_val_transformed)
xg_prediction = np.expm1(xg_prediction_log) 


mse = mean_squared_error(y_val, xg_prediction)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, xg_prediction)

print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

Root Mean Squared Error: 28261.89
R2 Score: 0.8959


cross validation

In [81]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xg_model, x_train_transformed, y_train_log, scoring="neg_mean_squared_error", cv=5)
rmse = np.sqrt(-scores.mean())
rmse

0.12537087985608542

FEATURE IMPORTANCE


In [82]:
import matplotlib.pyplot as plt

feature_names = full_pipe.get_feature_names_out()

importances = xg_model.feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(10))

                   feature  importance
43            num__QualxSF    0.222419
206      cat__CentralAir_Y    0.082918
205      cat__CentralAir_N    0.053316
25         num__GarageCars    0.037473
47       num__TotalQuality    0.035595
52        cat__MSZoning_RM    0.035527
266        ord__GarageCond    0.034200
45        num__AreaQuality    0.025666
138  cat__RoofMatl_CompShg    0.022236
37          num__TotalBath    0.019614


In [83]:
final_preds_log = xg_model.predict(test_transformed)
final_preds = np.expm1(final_preds_log)

final_preds = np.nan_to_num(final_preds, 
                             nan=np.nanmedian(train['SalePrice']), 
                             posinf=np.nanmax(train['SalePrice']))

submission = pd.DataFrame({
    "Id": test["Id"].astype(int),
    "SalePrice": final_preds
})

submission.to_csv("submission_final.csv", index=False)
